In [1]:
# 检查 VQGAN Stage 1 Checkpoint 的结构
# 以推断模型配置参数 (n_embed, embed_dim, z_channels, ch, ch_mult 等等)

import torch

CKPT_PATH = "vqgan/stage1.ckpt"

def analyze_vqgan_ckpt(path):
    print(f"正在分析: {path} ...")
    try:
        sd = torch.load(path, map_location="cpu")
        if "state_dict" in sd:
            sd = sd["state_dict"]
            print("✅ 这是一个 PyTorch Lightning Checkpoint (包含 'state_dict')")
        else:
            print("✅ 这是一个原生 PyTorch State Dict")
            
        print("-" * 30)
        
        # --- 侦探工作开始 ---
        
        # 1. 寻找 Codebook (量化器)
        # 通常 key 叫做 'quantize.embedding.weight'
        if 'quantize.embedding.weight' in sd:
            shape = sd['quantize.embedding.weight'].shape
            n_embed = shape[0]
            embed_dim = shape[1]
            print(f"🕵️ [关键线索] Codebook:")
            print(f"   - n_embed (词表大小): {n_embed}")
            print(f"   - embed_dim (嵌入维度): {embed_dim}")
        else:
            print("⚠️ 未找到 quantize.embedding.weight，可能是旧版模型或键名不同。")

        # 2. 寻找 Z-Channels (潜空间通道数)
        # 通常在 'post_quant_conv' 或 'quant_conv'
        if 'post_quant_conv.weight' in sd:
            # shape: [embed_dim, z_channels, 1, 1]
            shape = sd['post_quant_conv.weight'].shape
            z_channels = shape[1]
            print(f"🕵️ [关键线索] Z-Channels (Latent Channels): {z_channels}")
            
            if shape[0] != embed_dim:
                 print(f"⚠️ 注意: post_quant_conv 输出 ({shape[0]}) 与 embed_dim ({embed_dim}) 不匹配，请检查配置。")

        # 3. 寻找 Base Channels (基础通道数)
        # 也就是 model 的 ch 参数
        if 'decoder.conv_in.weight' in sd:
            # shape: [ch, z_channels, 3, 3]
            shape = sd['decoder.conv_in.weight'].shape
            ch = shape[0]
            print(f"🕵️ [关键线索] Base Channels (ch): {ch}")

        # 4. 推测 Downsampling 层数 (ch_mult)
        # 我们通过计算 decoder 中 up 层的数量来推测
        up_blocks = [k for k in sd.keys() if 'decoder.up' in k and '.block' in k]
        # decoder.up.0, decoder.up.1 ... 最大的数字
        if up_blocks:
            max_idx = 0
            for k in up_blocks:
                parts = k.split('.')
                try:
                    idx = int(parts[2]) # decoder.up.X...
                    if idx > max_idx: max_idx = idx
                except: pass
            # 层数通常是 max_index + 1
            print(f"🕵️ [推测线索] Downsample Levels (可能对应 ch_mult 长度): {max_idx + 1}")

        # 5. 检查输入通道
        if 'encoder.conv_in.weight' in sd:
             in_ch = sd['encoder.conv_in.weight'].shape[1]
             print(f"🕵️ [关键线索] 输入图片通道数 (in_channels): {in_ch} (3代表RGB)")

    except Exception as e:
        print(f"❌ 读取失败: {e}")

analyze_vqgan_ckpt(CKPT_PATH)

正在分析: vqgan/stage1.ckpt ...


C:\Users\qingy\AppData\Local\Temp\ipykernel_7384\243957873.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(path, map_location="cpu")


✅ 这是一个 PyTorch Lightning Checkpoint (包含 'state_dict')
------------------------------
🕵️ [关键线索] Codebook:
   - n_embed (词表大小): 8192
   - embed_dim (嵌入维度): 4
🕵️ [关键线索] Z-Channels (Latent Channels): 4
🕵️ [关键线索] Base Channels (ch): 128
🕵️ [推测线索] Downsample Levels (可能对应 ch_mult 长度): 3
🕵️ [关键线索] 输入图片通道数 (in_channels): 1 (3代表RGB)


In [2]:
import torch
import sys
import os

# 你的权重路径
CKPT_PATH = r"vqgan/stage1.ckpt"

def inspect_checkpoint():
    if not os.path.exists(CKPT_PATH):
        print(f"❌ 找不到文件: {CKPT_PATH}")
        return

    print(f"📂 正在加载: {CKPT_PATH} ...")
    try:
        ckpt = torch.load(CKPT_PATH, map_location="cpu")
    except Exception as e:
        print(f"❌ 加载失败: {e}")
        return

    # 1. 找到 state_dict
    if "state_dict" in ckpt:
        sd = ckpt["state_dict"]
        print("✅ 发现 'state_dict' 键")
    elif "model" in ckpt:
        sd = ckpt["model"]
        print("✅ 发现 'model' 键")
    else:
        sd = ckpt
        print("ℹ️ 直接使用 checkpoint 作为 state_dict")

    # 2. 打印前 20 个 Key 来观察前缀
    keys = list(sd.keys())
    print(f"\n🔑 总共有 {len(keys)} 个参数张量。")
    print("-" * 40)
    print("前 20 个 Key (请观察是否有前缀，如 'module.' 或 'autoencoder.'):")
    for k in keys[:20]:
        print(f"  {k}")
    print("-" * 40)

    # 3. 维度检查 (验证是否真的是 3D)
    # 尝试找一个卷积层
    conv_keys = [k for k in keys if "conv" in k and "weight" in k]
    if conv_keys:
        sample_key = conv_keys[0]
        shape = sd[sample_key].shape
        print(f"📐 维度抽查 [{sample_key}]:")
        print(f"   Shape: {shape}")
        if len(shape) == 5: # [Out, In, D, H, W]
            print("   ✅ 确认: 这是 3D 卷积权重")
        elif len(shape) == 4: # [Out, In, H, W]
            print("   ⚠️ 警告: 这是 2D 卷积权重! (如果你认为是 3D，可能拿错文件了)")
        else:
            print(f"   ℹ️ 未知维度: {len(shape)}D")
    
    # 4. Decoder 关键层检查
    print("\n🔍 Decoder 存在性检查:")
    decoder_keys = [k for k in keys if "decoder" in k]
    if len(decoder_keys) == 0:
        print("   ❌ 严重警告: 权重文件中没有包含 'decoder' 字样的 Key！")
        print("   可能原因: 这是一个只有 Encoder 的权重，或者是命名完全不同的模型。")
    else:
        print(f"   ✅ 包含 {len(decoder_keys)} 个 Decoder 参数。")
        print(f"   示例: {decoder_keys[0]}")

if __name__ == "__main__":
    inspect_checkpoint()

📂 正在加载: vqgan/stage1.ckpt ...
✅ 发现 'state_dict' 键

🔑 总共有 130 个参数张量。
----------------------------------------
前 20 个 Key (请观察是否有前缀，如 'module.' 或 'autoencoder.'):
  encoder.conv_in.weight
  encoder.conv_in.bias
  encoder.down.0.block.0.norm1.weight
  encoder.down.0.block.0.norm1.bias
  encoder.down.0.block.0.conv1.weight
  encoder.down.0.block.0.conv1.bias
  encoder.down.0.block.0.norm2.weight
  encoder.down.0.block.0.norm2.bias
  encoder.down.0.block.0.conv2.weight
  encoder.down.0.block.0.conv2.bias
  encoder.down.0.downsample.conv.weight
  encoder.down.0.downsample.conv.bias
  encoder.down.1.block.0.norm1.weight
  encoder.down.1.block.0.norm1.bias
  encoder.down.1.block.0.conv1.weight
  encoder.down.1.block.0.conv1.bias
  encoder.down.1.block.0.norm2.weight
  encoder.down.1.block.0.norm2.bias
  encoder.down.1.block.0.conv2.weight
  encoder.down.1.block.0.conv2.bias
----------------------------------------
📐 维度抽查 [encoder.conv_in.weight]:
   Shape: torch.Size([64, 1, 3, 3, 3])
   ✅ 确认:

C:\Users\qingy\AppData\Local\Temp\ipykernel_7384\2256192671.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CKPT_PATH, map_location="cpu")
